# Лабораторная 09. Cache и память

Цель: понять, когда cache помогает, а когда просто занимает memory.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName('lab-09-cache').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
base_uri = Path('spark_core_data').absolute().as_uri()
orders = spark.read.parquet(f'{base_uri}/orders')
print('Spark UI:', spark.sparkContext.uiWebUrl)

## Без cache
Один и тот же filtered DataFrame используется несколько раз. Spark может пересчитывать lineage для каждой action.

In [ ]:
paid = orders.filter(F.col('status') == 'paid').select('order_id', 'customer_id', 'order_amount')
paid.count()
paid.groupBy('customer_id').count().count()

## С cache
`cache()` только помечает DataFrame. Чтобы реально положить данные в cache, нужна action.

In [ ]:
paid_cached = paid.cache()
paid_cached.count()  # materialization
print('Откройте Storage tab:', spark.sparkContext.uiWebUrl)

In [ ]:
paid_cached.groupBy('customer_id').count().count()
paid_cached.agg(F.sum('order_amount')).show()

## Очистка cache

In [ ]:
paid_cached.unpersist()

Вопросы:

- Почему после `cache()` нужно вызвать action?
- Почему cache помогает только при повторном использовании?
- Что видно на Storage tab?
- Что будет, если закэшировать слишком большой DataFrame?
- Когда cache лучше не использовать?